# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 实践实验：用于基于内容的过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，构建一个电影推荐系统。 

# 大纲 <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [1 - 软件包](#1)
- [2 - 电影评分数据集](#2)
  - [2.1 使用神经网络实现基于内容的过滤](#2.1)
  - [2.2 准备训练数据](#2.2)
- [3 - 用于基于内容的过滤的神经网络](#3)
  - [3.1 预测](#3.1)
    - [练习 1](#ex01)
- [4 - 恭喜！](#4)

<a name="1"></a>
## 1 - 软件包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的软件包 NumPy、TensorFlow，以及 [scikit-learn](https://scikit-learn.org/stable/) 中的一些实用函数。我们还会使用 [tabulate](https://pypi.org/project/tabulate/) 来美观地打印表格，使用 [Pandas](https://pandas.pydata.org/) 来组织表格数据。

In [1]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。 

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 位用户对 9000 部电影的评分，评分范围为 0.5 到 5，以 0.5 为步长递增。数据集经过缩减，聚焦于 2000 年以后的电影和热门类型。缩减后的数据集包含 $n_u = 395$ 位用户和 $n_m= 694$ 部电影。对每部电影，数据集提供了电影名称、上映日期以及一个或多个类型。例如，“Toy Story 3”于 2010 年上映，有多个类型：“Adventure|Animation|Children|Comedy|Fantasy|IMAX”。 除了评分之外，该数据集几乎不包含关于用户的其他信息。该数据集用于为下文描述的神经网络创建训练向量。 

<a name="2.1"></a>
### 2.1 使用神经网络实现基于内容的过滤

在协同过滤实验中，你生成了两个向量——用户向量和物品/电影向量，它们的点积可以用来预测评分。这些向量完全由评分推导而来。   

基于内容的过滤同样会生成用户向量和电影特征向量，但它会利用关于用户和/或电影的其他可用信息来改进预测。这些附加信息会被输入到一个神经网络中，由该网络生成用户向量和电影向量，如下所示。
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>
输入网络的电影内容包括原始数据和一些“工程特征”。回想一下课程 1 第 2 周实验 4 中关于特征工程的讨论和实验。原始特征是电影的上映年份，以及以独热向量形式表示的电影类型。共有 14 种类型。工程特征是从用户评分推导出的平均评分。对于有多个类型的电影，每个类型都有一个对应的训练向量。 

用户内容则完全由工程特征组成。对每个用户计算其按类型划分的平均评分。此外，还有用户 id、评分次数和平均评分可用，但它们不包含在训练或预测的输入中。它们在解读数据时很有用。

训练集由数据集中用户做出的所有评分组成。用户向量和电影/物品向量会一起作为训练集输入上述网络。对于某用户评过的所有电影，用户向量都是相同的。 

下面，让我们加载并展示部分数据。

In [2]:
# 加载数据，设置配置变量
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # 训练期间去掉 userid、评分数量和平均评分
num_item_features = item_train.shape[1] - 1  # 训练时去掉电影 ID
uvs = 3  # 用户类型向量起点
ivs = 3  # 物品类型向量起点
u_s = 3  # 训练所用列的起点（用户）
i_s = 1  # 训练所用列的起点（物品）
scaledata = True  # 若为 True，则对数据应用标准化
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 58187


部分用户和物品/电影特征没有用于训练。下面，方括号“[]”中的特征，例如“user id”“rating count”和“rating ave”，在模型训练和使用时并不包含在内。注意，对于某用户评过的所有电影，用户向量都是相同的。

In [3]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9


In [4]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8798,2004,3.8,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [5]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  4.  4.  3.5 3.5]


从上面可以看到，电影 6874 是一部 2003 年上映的动作片。用户 2 对动作片的平均评分为 3.9。此外，电影 6874 还被列入 Crime（犯罪）和 Thriller（惊悚）类型。MovieLens 用户给这部电影打出的平均评分为 4。一个训练样本由两张表中的各一行以及 y_train 中的一个评分组成。

<a name="2.2"></a>
### 2.2 准备训练数据
回想一下课程 1 第 2 周的内容，你曾探索过将特征缩放作为改善收敛的手段。我们将使用 [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 对输入特征进行缩放。这在课程 1 第 2 周的实验 5 中用到过。下面还展示了 inverse_transform，用于还原出原始输入。

In [6]:
# 对训练数据做缩放
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

True
True


为了能够评估结果，我们将按照课程 2 第 3 周讨论过的方法，把数据划分为训练集和测试集。这里我们将使用 [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来划分并打乱数据。注意，将初始随机状态设置为相同的值可以确保 item、user 和 y 被完全一致地打乱。

In [7]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (46549, 17)
movie/item test  data shape: (11638, 17)


缩放并打乱后的数据现在均值为零。

In [8]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,0.6,0.7,0.6,0.6,0.7,0.7,0.5,0.7,0.2,0.3,0.3,0.5,0.5,0.8,0.5
0,0,1.6,1.5,1.7,0.9,1.0,1.4,0.8,-1.2,1.2,1.2,1.6,0.9,1.4,1.2,1.0
0,0,0.8,0.6,0.7,0.5,0.6,0.6,0.3,-1.2,0.7,0.8,0.9,0.6,0.2,0.6,0.6
1,0,-0.1,0.2,-0.1,0.3,0.7,0.3,0.2,1.0,-0.5,-0.7,-2.1,0.5,0.7,0.3,0.0
-1,0,-1.3,-0.8,-0.8,0.1,-0.1,-1.1,-0.9,-1.2,-1.5,-0.6,-0.5,-0.6,-0.9,-0.4,-0.9


使用 Min Max Scaler 对目标评分进行缩放，将目标缩放到 -1 到 1 之间。我们使用 scikit-learn，因为它带有 inverse_transform。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [9]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

(46549, 1) (11638, 1)


<a name="3"></a>
## 3 - 用于基于内容的过滤的神经网络
现在，让我们按照上图描述的结构构建一个神经网络。它将包含两个网络，通过点积组合在一起。你将构建这两个网络。在本例中，它们是完全相同的。注意，这两个网络并不一定要相同。如果用户内容远比电影内容庞大，你可以选择让用户网络的复杂度高于电影网络。在这里，两者内容规模相近，因此两个网络是相同的。

- 使用 Keras 序贯（sequential）模型
    - 第一层是一个 Dense 层，有 256 个单元，relu 激活。
    - 第二层是一个 Dense 层，有 128 个单元，relu 激活。
    - 第三层是一个 Dense 层，有 `num_outputs` 个单元，线性激活或无激活。   
    
网络的其余部分会提供给你。提供的代码没有使用 Keras 序贯模型，而是使用了 Keras 的 [functional api](https://keras.io/guides/functional_api/)。这种格式在组件如何相互连接方面提供了更大的灵活性。

In [10]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###   
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    ### END CODE HERE ###  
])

# 创建用户输入，并指向基础网络
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# 创建物品输入，并指向基础网络
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# 计算向量 vu 和 vm 的点积
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# 指定模型的输入和输出
model = Model([input_user, input_item], output)

model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 14)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 16)]         0                                            
__________________________________________________________________________________________________
sequential (Sequential)         (None, 32)           40864       input_1[0][0]                    
__________________________________________________________________________________________________
sequential_1 (Sequential)       (None, 32)           41376       input_2[0][0]                    
______________________________________________________________________________________________

In [11]:
# 公共测试
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  你可以按照如下方式创建一个带 relu 激活的 Dense 层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看解答</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    

我们将使用均方误差损失和 Adam 优化器。

In [12]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [13]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

Train on 46549 samples
Epoch 1/30
46549/46549 [==============================] - 6s 133us/sample - loss: 0.1254
Epoch 2/30
46549/46549 [==============================] - 6s 122us/sample - loss: 0.1187
Epoch 3/30
46549/46549 [==============================] - 6s 121us/sample - loss: 0.1169
Epoch 4/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1154
Epoch 5/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1142
Epoch 6/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1130
Epoch 7/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1119
Epoch 8/30
46549/46549 [==============================] - 6s 119us/sample - loss: 0.1110
Epoch 9/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1095
Epoch 10/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1083
Epoch 11/30
46549/46549 [==============================] - 6s 122us/sample - loss: 0.1

评估模型以确定其在测试数据上的损失。它与训练损失相当，表明模型没有对训练数据产生明显的过拟合。

In [14]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

11638/11638 [==============================] - 0s 36us/sample - loss: 0.1045


0.10449595100221243

<a name="3.1"></a>
### 3.1 预测
下面，你将在多种情形下使用你的模型进行预测。 
#### 为新用户进行预测
首先，我们将创建一个新用户，并让模型为该用户推荐电影。在你对这个示例用户内容尝试过之后，欢迎把用户内容改成你自己的偏好，看看模型会推荐什么。注意，评分介于 0.5 和 5.0 之间（含端点），以半分为步长递增。

In [15]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])


让我们看看这个新用户评分最高的电影。回想一下，该用户向量的类型偏好偏向喜剧（Comedy）和爱情（Romance）。
下面，我们将使用一组电影/物品向量 `item_vecs`，其中为训练集/测试集中的每部电影都提供了一个向量。它将与上面的用户向量相匹配，缩放后的向量用于为我们的新用户预测所有电影的评分。

In [16]:
# 生成并复制用户向量，使其数量与数据集中的电影数一致。
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# 对向量做缩放，并对所有电影进行预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

y_p,movie id,rating ave,title,genres
4.86762,64969,3.61765,Yes Man (2008),Comedy
4.86692,69122,3.63158,"Hangover, The (2009)",Comedy|Crime
4.86477,63131,3.625,Role Models (2008),Comedy
4.85853,60756,3.55357,Step Brothers (2008),Comedy
4.85785,68135,3.55,17 Again (2009),Comedy|Drama
4.85178,78209,3.55,Get Him to the Greek (2010),Comedy
4.85138,8622,3.48649,Fahrenheit 9/11 (2004),Documentary
4.8505,67087,3.52941,"I Love You, Man (2009)",Comedy
4.85043,69784,3.65,Brüno (Bruno) (2009),Comedy
4.84934,89864,3.63158,50/50 (2011),Comedy|Drama


如果你确实在上面创建了一个用户，值得注意的是，该网络是在给定包含**一组**用户类型评分的用户向量的情况下训练出来预测用户评分的。如果不存在具有类似评分组合的用户，仅为单一类型给出最高评分、而其余类型给出最低评分，对网络来说可能并没有意义。

#### 为已有用户进行预测。
让我们看看数据集中的一位用户“user 36”的预测结果。我们可以将预测评分与该用户的实际评分进行比较。注意，具有多个类型的电影会在训练数据中出现多次。例如，“The Time Machine”有三个类型：Adventure、Action、Sci-Fi

In [17]:
uid =  36 
# 构造一组用户向量。这是同一个向量经变换后重复得到的。
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# 对向量做缩放，并对所有电影进行预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#打印排序后的预测结果
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

y_p,y,user,user genre ave,movie rating ave,title,genres
3.1,3.0,36,3.00,2.86,"Time Machine, The (2002)",Adventure
3.0,3.0,36,3.00,2.86,"Time Machine, The (2002)",Action
2.8,3.0,36,3.00,2.86,"Time Machine, The (2002)",Sci-Fi
2.3,1.0,36,1.00,4.00,"Beautiful Mind, A (2001)",Romance
2.2,1.0,36,1.50,4.00,"Beautiful Mind, A (2001)",Drama
1.6,1.5,36,1.75,3.52,Road to Perdition (2002),Crime
1.6,2.0,36,1.75,3.52,Gangs of New York (2002),Crime
1.5,1.5,36,1.50,3.52,Road to Perdition (2002),Drama
1.5,2.0,36,1.50,3.52,Gangs of New York (2002),Drama


#### 寻找相似物品
上面的神经网络生成了两个特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。它们是包含 32 个元素的向量，其数值难以直接解读。不过，相似的物品会拥有相似的向量。这一信息可以用来生成推荐。例如，如果某个用户给“Toy Story 3”打了高分，就可以通过选取具有相似电影特征向量的电影来推荐类似的电影。

一种相似度度量是两个向量 $ \mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### 练习 1

编写一个函数来计算平方距离。

In [20]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    d = sum(np.square(a-b))
    ### END CODE HERE ###     
    return (d)

In [21]:
# 公共测试
test_sq_dist(sq_dist)

All tests passed!


In [22]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  求和通常意味着应该使用 for 循环，但在这里，减法可以在一条语句中按元素完成。此外，你可以使用 np.square 对减法的结果按元素求平方。np.sum 可以用来对平方后的元素求和。
    
</details>

    

电影之间的距离矩阵可以在模型训练完成后计算一次，之后在生成新推荐时重复使用，而无需重新训练。模型训练完成后的第一步，是获得每部电影的特征向量 $v_m$。为此，我们将使用训练好的 `item_NN`，并构建一个小模型，让我们能够将电影向量输入其中生成 $v_m$。

In [23]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # 输入层
vm_m = item_NN(input_item_m)                                       # 使用训练好的 item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # 按原始模型中的做法加入归一化
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 16)]         0                                            
__________________________________________________________________________________________________
sequential_1 (Sequential)       (None, 32)           41376       input_3[0][0]                    
__________________________________________________________________________________________________
tf_op_layer_l2_normalize_2/Squa [(None, 32)]         0           sequential_1[1][0]               
__________________________________________________________________________________________________
tf_op_layer_l2_normalize_2/Sum  [(None, 1)]          0           tf_op_layer_l2_normalize_2/Square
____________________________________________________________________________________________

有了电影模型之后，你可以把一组物品/电影向量作为输入，用该模型进行预测，从而创建一组电影特征向量。`item_vecs` 是所有电影向量的集合。回想一下，同一部电影会按它的每个类型分别出现为一个向量。它必须经过缩放才能与训练好的模型一起使用。预测的结果是每部电影的一个 32 元素特征向量。

In [24]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

size of all predicted movie feature vectors: (1883, 32)


现在让我们计算每部电影特征向量与其他所有电影特征向量之间平方距离的矩阵：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后，我们可以通过在每一行中找出最小值来找到最相似的电影。我们将利用 [numpy masked arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html) 来避免选中同一部电影。对角线上被掩蔽的值不会参与计算。

In [25]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # 把对角线屏蔽掉

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama,John Q (2002),Drama
Save the Last Dance (2001),Romance,Saving Silverman (Evil Woman) (2001),Romance
"Wedding Planner, The (2001)",Comedy,National Lampoon's Van Wilder (2002),Comedy
"Wedding Planner, The (2001)",Romance,Mr. Deeds (2002),Romance
Hannibal (2001),Horror,Final Destination 2 (2003),Horror
Hannibal (2001),Thriller,"Sum of All Fears, The (2002)",Thriller
Saving Silverman (Evil Woman) (2001),Comedy,Cats & Dogs (2001),Comedy
Saving Silverman (Evil Woman) (2001),Romance,Save the Last Dance (2001),Romance
Down to Earth (2001),Comedy,Joe Dirt (2001),Comedy
Down to Earth (2001),Fantasy,"Haunted Mansion, The (2003)",Fantasy


结果表明，模型推荐的电影来自同一类型。

<a name="4"></a>
## 4 - 恭喜！ <img align="left" src="./images/film_award.png" style=" width:40px;">
你已经完成了一个基于内容的推荐系统。    

这种结构是许多商业推荐系统的基础。如果有可用信息，用户内容可以大幅扩展，以纳入更多关于用户的信息。物品也不限于电影。它可以用来推荐任何物品——书籍、汽车，或者与你“购物车”中的物品相似的商品。